In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [4]:
import pandas as pd

# Nota la 'r' al inicio
PATH = r"C:\Users\Brian OAQ\Desktop\OAQ\2026\12. Manuscrito de un artículo Modelo predictivo de nubosidad para observaciones astronómicas\OAQ-AstroForecast\data\raw\ambient_weather\meteorologia_2026.csv"

df = pd.read_csv(PATH)
print(df.shape)
df.head()

(61697, 8)


,station_mac,fecha_utc,temp_f,humidity,wind_speed_mph,uv,solar_radiation,rain_daily_in
0,F4:CF:A2:E2:BC:AD,2026-01-01 00:00:00,60.8,68.0,0.0,0.0,0.0,0.0
1,F4:CF:A2:E2:BC:AD,2026-01-01 00:05:00,60.8,69.0,0.2,0.0,0.0,0.0
2,F4:CF:A2:E2:BC:AD,2026-01-01 00:10:00,60.6,71.0,1.8,0.0,0.0,0.0
3,F4:CF:A2:E2:BC:AD,2026-01-01 00:15:00,60.4,72.0,0.7,0.0,0.0,0.0
4,F4:CF:A2:E2:BC:AD,2026-01-01 00:20:00,60.3,73.0,0.0,0.0,0.0,0.0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61697 entries, 0 to 61696
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   station_mac      61697 non-null  object 
 1   fecha_utc        61697 non-null  object 
 2   temp_f           61581 non-null  float64
 3   humidity         61581 non-null  float64
 4   wind_speed_mph   61581 non-null  float64
 5   uv               61578 non-null  float64
 6   solar_radiation  61578 non-null  float64
 7   rain_daily_in    61697 non-null  float64
dtypes: float64(6), object(2)
memory usage: 3.8+ MB


In [6]:
df.columns.tolist()

['station_mac',
 'fecha_utc',
 'temp_f',
 'humidity',
 'wind_speed_mph',
 'uv',
 'solar_radiation',
 'rain_daily_in']

In [7]:
df["fecha_utc"] = pd.to_datetime(df["fecha_utc"])

print(df["fecha_utc"].min())
print(df["fecha_utc"].max())

2026-01-01 00:00:00
2026-04-29 21:00:00


In [8]:
df["station_mac"].value_counts()

station_mac
F4:CF:A2:E2:C0:8B    32077
F4:CF:A2:E2:BC:AD    29620
Name: count, dtype: int64

In [9]:
missing = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      * 100
)

missing

solar_radiation    0.192878
uv                 0.192878
humidity           0.188016
temp_f             0.188016
wind_speed_mph     0.188016
fecha_utc          0.000000
station_mac        0.000000
rain_daily_in      0.000000
dtype: float64

In [10]:
duplicates = df.duplicated().sum()

print(f"Duplicados: {duplicates}")

Duplicados: 0


In [11]:
for station in df["station_mac"].unique():

    tmp = (
        df[df["station_mac"] == station]
        .sort_values("fecha_utc")
    )

    delta = (
        tmp["fecha_utc"]
        .diff()
        .dt.total_seconds()
        .dropna()
    )

    print("\n")
    print(station)
    print(delta.value_counts().head())



F4:CF:A2:E2:BC:AD
fecha_utc
300.0    29393
240.0       61
360.0       56
600.0       28
900.0       17
Name: count, dtype: int64


F4:CF:A2:E2:C0:8B
fecha_utc
300.0    31939
240.0       41
360.0       39
600.0       16
900.0        7
Name: count, dtype: int64


In [12]:
summary = (
    df.groupby("station_mac")
      .agg({
          "temp_f":["min","max","mean"],
          "humidity":["min","max","mean"],
          "wind_speed_mph":["min","max","mean"],
          "solar_radiation":["min","max","mean"]
      })
)

summary

temp_f                  humidity                   \
                     min   max       mean      min   max       mean   
station_mac                                                           
F4:CF:A2:E2:BC:AD   44.8  78.4  57.691343     24.0  99.0  84.142717   
F4:CF:A2:E2:C0:8B   47.8  74.7  57.197613     22.0  99.0  81.862497   

                  wind_speed_mph                 solar_radiation           \
                             min   max      mean             min      max   
station_mac                                                                 
F4:CF:A2:E2:BC:AD            0.0   8.7  0.331916             0.0  2358.15   
F4:CF:A2:E2:C0:8B            0.0  18.3  2.434768             0.0  1390.72   

                               
                         mean  
station_mac                    
F4:CF:A2:E2:BC:AD  134.604710  
F4:CF:A2:E2:C0:8B  146.116426

In [13]:
summary_count = (
    df.groupby("station_mac")
      .count()
)

summary_count

,fecha_utc,temp_f,humidity,wind_speed_mph,uv,solar_radiation,rain_daily_in
station_mac,,,,,,,
F4:CF:A2:E2:BC:AD,29620,29618,29618,29618,29618,29618,29620
F4:CF:A2:E2:C0:8B,32077,31963,31963,31963,31960,31960,32077
